# 🪪 Notebook 3: OAuth 2.0 Authorization Code Flow

**The problem:** the user wants to let *AcmePhoto* see their Google Photos — **without** giving AcmePhoto their Google password.

OAuth solves this with a careful dance between four roles:

- **Resource Owner** — the user.
- **Client** — the app that wants access (AcmePhoto).
- **Authorization Server** — issues access tokens (Google's login screen).
- **Resource Server** — holds the data (the Google Photos API).

We will simulate the **authorization code** flow with two tiny FastAPI apps: one is the auth server, one is the client app. We will run them in the notebook with `httpx` for the HTTP calls.

## Learning objectives
- See the *redirect → consent → code → token → API call* sequence step by step.
- Understand why we exchange a *code* for a *token* (instead of returning the token directly).

## 🛠️ Setup

```bash
cd 01-foundations/authentication-authorization
uv sync
```

This is pure Python; no external services. We use FastAPI's in-process test client so we don't have to start real servers.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
import secrets

# -----------------------------------------------------------------------------
# Authorization Server (think: Google)
# -----------------------------------------------------------------------------
auth_server = FastAPI()
USERS = {"alice": "hunter2"}                # user db
CLIENTS = {"acmephoto": "client-secret"}    # registered third-party apps
CODES = {}                                  # short-lived auth codes
TOKENS = {}                                 # access tokens -> user

@auth_server.post("/authorize")
def authorize(username: str, password: str, client_id: str, redirect_uri: str):
    # The user is logging in to Google to give consent. In a real flow this
    # would be a browser page, not a JSON POST.
    if USERS.get(username) != password:
        raise HTTPException(401, "bad creds")
    if client_id not in CLIENTS:
        raise HTTPException(400, "unknown client")
    code = secrets.token_urlsafe(8)
    CODES[code] = {"user": username, "client_id": client_id}
    # The auth server "redirects" the browser back to the client with this code.
    return {"redirect_to": f"{redirect_uri}?code={code}"}

@auth_server.post("/token")
def token(code: str, client_id: str, client_secret: str):
    # The client now exchanges the code for an access token, proving its identity
    # with its client_secret. Notice this happens server-to-server, NOT in the browser.
    if CLIENTS.get(client_id) != client_secret:
        raise HTTPException(401, "bad client")
    info = CODES.pop(code, None)
    if not info or info["client_id"] != client_id:
        raise HTTPException(400, "bad code")
    access_token = secrets.token_urlsafe(16)
    TOKENS[access_token] = info["user"]
    return {"access_token": access_token}

# -----------------------------------------------------------------------------
# Resource Server (think: Google Photos API)
# -----------------------------------------------------------------------------
resource_server = FastAPI()

@resource_server.get("/photos")
def photos(access_token: str):
    user = TOKENS.get(access_token)
    if not user:
        raise HTTPException(401, "bad token")
    return {"user": user, "photos": ["beach.jpg", "cat.jpg"]}

auth_client = TestClient(auth_server)
res_client = TestClient(resource_server)
print("FastAPI apps ready ✅")

In [ ]:
# -----------------------------------------------------------------------------
# THE FLOW (annotated step by step)
# -----------------------------------------------------------------------------

# Step 1: AcmePhoto sends Alice to the auth server with its client_id and a
# redirect_uri. Alice logs in there and clicks "Allow".
r = auth_client.post("/authorize", params={
    "username": "alice", "password": "hunter2",
    "client_id": "acmephoto",
    "redirect_uri": "https://acmephoto.example/callback",
})
redirect = r.json()["redirect_to"]
print("Step 1 — auth server redirects browser to:", redirect)

# Step 2: AcmePhoto's backend extracts the `code` from the redirect URL.
code = redirect.split("code=")[1]
print("Step 2 — AcmePhoto received code:", code)

# Step 3: AcmePhoto exchanges the code for an access_token, proving it really
# is AcmePhoto using its client_secret. This is a back-channel call.
r = auth_client.post("/token", params={
    "code": code,
    "client_id": "acmephoto",
    "client_secret": "client-secret",
})
access_token = r.json()["access_token"]
print("Step 3 — got access_token:", access_token)

# Step 4: AcmePhoto calls the Resource Server with the token to actually
# fetch Alice's photos. AcmePhoto NEVER saw Alice's password.
r = res_client.get("/photos", params={"access_token": access_token})
print("Step 4 — photos:", r.json())

## 🧠 Why a *code* and not the token directly?

You might wonder: why bother with a one-time code? Why doesn't the auth server just hand the access token to the browser?

Because **the browser is a hostile environment**: extensions, redirects, browser history, screen-sharing, etc. The code is harmless on its own — only AcmePhoto's backend (which knows the `client_secret`) can swap it for a real token. The token never touches the browser.

This is also why mobile/SPA apps use **PKCE** — a slightly different variant that doesn't require a server-held secret. Same idea: don't expose the powerful credential.

## ✅ Recap

The OAuth dance trades a tiny bit of complexity for a huge security win: **third-party apps can act on behalf of the user without ever seeing the password**, and the user can revoke access at any time.